# Poison dart frogs

Can you estimate how toxic a poison dart frog is based on its brightness?

Poison dart frogs are small, colorful frogs from Central and South America. They're famous for their toxic skin, which some indigenous people use for blow darts.

Scientists studied the connection between the bright colors and toxicity of the poison dart frog **Dendrobates pumilio**. While they're usually red with blue legs, they can be found in about 15 different colors in Panama.

The file `frog-reflectance.csv` contains the average brightness (from 0 to 1) of 10 frog populations, measured on both the back and belly.

The file `frog-toxicity.csv` provides a `toxicity` score (from 0 to 1) for 48 frogs from the same 10 populations. Toxicity was tested by injecting frog skin extracts into mice and recording how long it took them to fall back asleep. Longer times suggest higher `toxicity`.

In [1]:
# FOR GOOGLE COLAB ONLY.
# Uncomment and run the code below. A dialog will appear to upload files.
# Upload 'frog-reflectance.csv', 'frog-toxicity.csv', and 'linear_model.py'.

# from google.colab import files
# uploaded = files.upload()

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from linear_model import LinearModel

df = pd.read_csv('frog-reflectance.csv')
df.head()

,pop_id,back_brightness,belly_brightness
0,AG,0.43,0.46
1,AL,0.64,0.42
2,BCG,1.00,0.95
3,BCO,0.82,1.00
4,CA,0.46,0.51


In [3]:
tox = pd.read_csv('frog-toxicity.csv')
tox.head()

,pop_id,frog_id,toxicity
0,BCO,BC167,0.886925
1,BCO,BC166,0.803917
2,BCO,BC171,0.625008
3,BCO,BC169,0.975847
4,BCO,BC165,0.774033


# Project Ideas

- Build two linear models:
	- Model 1: `toxicity` (x-axis) vs. `back_brightness`
	
	- Model 2: `toxicity` vs. `belly_brightness`

- To create the models, first calculate the average `toxicity` for each `pop_id`, then merge these values into the main dataframe (`df`).

- Visualize the data by creating scatter plots with best-fit lines for both models.

- Compare the goodness of fit between back and belly brightness and explore possible biological explanations for any differences in correlation.


## Lösung

### Average toxicity per population

In [ ]:
import doctest


def average_toxicity(toxicity: pd.DataFrame) -> pd.DataFrame:
    """Liefert die durchschnittliche Giftigkeit pro Population als Tabelle mit pop_id und toxicity.

    >>> toy = pd.DataFrame({"pop_id": ["A", "A", "B"], "frog_id": ["a1", "a2", "b1"], "toxicity": [0.25, 0.75, 0.9]})
    >>> average_toxicity(toy).to_dict("records")
    [{'pop_id': 'A', 'toxicity': 0.5}, {'pop_id': 'B', 'toxicity': 0.9}]
    """
    return toxicity.groupby("pop_id", as_index=False)["toxicity"].mean()


avg_tox = average_toxicity(tox)
avg_tox

### Merge into the main dataframe

In [ ]:
def merge_toxicity(reflectance: pd.DataFrame, avg_toxicity: pd.DataFrame) -> pd.DataFrame:
    """Hängt die durchschnittliche Giftigkeit über pop_id an die Helligkeiten an.

    >>> refl = pd.DataFrame({"pop_id": ["A", "B"], "back_brightness": [0.4, 0.9]})
    >>> avg = pd.DataFrame({"pop_id": ["B", "A"], "toxicity": [0.8, 0.3]})
    >>> merge_toxicity(refl, avg).to_dict("records")
    [{'pop_id': 'A', 'back_brightness': 0.4, 'toxicity': 0.3}, {'pop_id': 'B', 'back_brightness': 0.9, 'toxicity': 0.8}]
    """
    # validate bricht ab, falls eine Population doppelt vorkommt
    return pd.merge(reflectance, avg_toxicity, on="pop_id", validate="one_to_one")


df = merge_toxicity(df, avg_tox)
df

### Linear models

- Model 1: `toxicity` (x-axis) vs. `back_brightness`
- Model 2: `toxicity` vs. `belly_brightness`

In [ ]:
def fit_model(data: pd.DataFrame, column: str) -> LinearModel:
    """Passt ein lineares Modell an: toxicity auf der x-Achse, die Helligkeit aus column auf der y-Achse.

    >>> toy = pd.DataFrame({"toxicity": [0.2, 0.6, 1.0], "back_brightness": [0.3, 0.5, 0.7]})
    >>> model = fit_model(toy, "back_brightness")
    >>> model.model_name, [round(float(value), 2) for value in (model.slope, model.intercept, model.rsquared)]
    ('back_brightness', [0.5, 0.2, 1.0])
    """
    return LinearModel(column).fit(data["toxicity"], data[column])


back_model = fit_model(df, "back_brightness")
belly_model = fit_model(df, "belly_brightness")
back_model.print_model_info()
print()
belly_model.print_model_info()

### Scatter plots with best-fit lines

In [ ]:
def plot_models(data: pd.DataFrame, models: list[LinearModel]) -> list[plt.Axes]:
    """Zeichnet für jedes Modell ein Streudiagramm mit Ausgleichsgerade nebeneinander, R² steht im Titel.

    >>> toy = pd.DataFrame({"pop_id": ["A", "B", "C"], "toxicity": [0.2, 0.6, 1.0],
    ...                     "back_brightness": [0.3, 0.5, 0.7], "belly_brightness": [0.4, 0.6, 0.5]})
    >>> axes = plot_models(toy, [fit_model(toy, "back_brightness"), fit_model(toy, "belly_brightness")])
    >>> [ax.get_title() for ax in axes]
    ['back_brightness (R² = 1.00)', 'belly_brightness (R² = 0.25)']
    >>> [(len(ax.collections), len(ax.lines), len(ax.texts)) for ax in axes]
    [(1, 1, 3), (1, 1, 3)]
    >>> plt.close(axes[0].figure)
    """
    fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4.5), sharex=True, sharey=True)
    x = data["toxicity"]

    for ax, model in zip(axes, models):
        y = data[model.model_name]
        ax.scatter(x, y, s=60, color="#c2410c", edgecolor="white", zorder=3)

        # Populationen beschriften, damit man Ausreißer zuordnen kann
        for pop_id, x_value, y_value in zip(data["pop_id"], x, y):
            ax.annotate(pop_id, (x_value, y_value), xytext=(5, 3), textcoords="offset points", fontsize=8, color="gray")

        # plot_model zeichnet mit plt.plot auf die aktuelle Achse, daher vorher umschalten
        plt.sca(ax)
        model.plot_model(x.min(), x.max())

        ax.set_title(f"{model.model_name} (R² = {model.rsquared:.2f})")
        ax.set_xlabel("toxicity")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    axes[0].set_ylabel("brightness")
    fig.suptitle("Giftigkeit gegen Helligkeit von Rücken und Bauch")
    fig.tight_layout()
    return list(axes)


plot_models(df, [back_model, belly_model])
plt.show()